# Iridium 1.0 Studio -- Kaggle

**The model is untrained until you run the training cell below.** Loading
this notebook, or running the setup cells, produces random weights and
nothing else. A checkpoint means something only after a training run has
actually completed.

**What a free session buys you** is printed by the dry-run cell (section
3) before anything trains: it prints `iridium.presets.preset_table()` and
`iridium.training.run_preset.dry_run()`'s cost/audit report, both computed
from `estimate_hours` and `iridium.training.budget.audit` -- not measured,
arithmetic, and optimistic (see the warning in `iridium/presets.py`). Read
those numbers before starting a long run; they tell you honestly whether
the chosen preset's step budget fits in the quota you have.

**Network is required.** The text/chat data path
(`iridium.training.datasets.build_corpus`) streams real text from the
Hugging Face `datasets` library, and the preset's subword tokenizer is
trained from that same stream the first time it runs. No network (or no
`datasets` package) means `train_preset` refuses to fall back to a
silently mismatched byte-level vocabulary and raises instead -- see
`iridium/training/run_preset.py`.

Default preset here: `chat-100m`. Change `PRESET` in section 2 to
train a different one; `python -m iridium presets` (or the table this
notebook prints) lists every option and what each needs.

## 1. Get the source (private repo)

Never prints or stores the token; see `notebooks/README.md` for how to add one.

In [ ]:
# This repository is private until Iridium 1.0 ships. This cell never
# hard-codes or prints a token: it reads one from the host's own secret
# store (or an env var on a plain Jupyter host) and hands it to git only
# through an environment-scoped header, so it never lands in .git/config.
import os, sys, subprocess
from pathlib import Path
token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    token = None

REPO_OWNER = 'sporadicstudiosind-cloud'
REPO_NAME = 'test'
RELEASE_BRANCH = 'claude/gallant-faraday-lhycva'
ROOT = Path.cwd()

if (ROOT / 'iridium' / 'presets.py').is_file():
    print('Already inside a checkout of the repository:', ROOT)
else:
    ROOT = Path.cwd() / REPO_NAME
    if ROOT.is_dir():
        print('Reusing existing checkout at', ROOT)
    elif token:
        # The token travels in an environment-scoped git config header:
        # never in the URL (git would store it in .git/config) and never in
        # the argv (a failed subprocess prints its argv).
        import base64
        basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
        env = dict(os.environ, GIT_CONFIG_COUNT='1',
                   GIT_CONFIG_KEY_0='http.https://github.com/.extraheader',
                   GIT_CONFIG_VALUE_0=f'AUTHORIZATION: basic {basic}',
                   GIT_TERMINAL_PROMPT='0')
        done = subprocess.run(['git', 'clone', '--depth', '1', '--branch', RELEASE_BRANCH,
                               f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git',
                               str(ROOT)], env=env, capture_output=True, text=True)
        del env, basic
        if done.returncode != 0:
            raise RuntimeError('git clone failed (exit %d). Check the token can read '
                               'the repository and the branch exists.' % done.returncode)
        print('Cloned', REPO_OWNER + '/' + REPO_NAME, '@', RELEASE_BRANCH, 'to', ROOT)
    else:
        print('No GITHUB_TOKEN found. In Kaggle: Add-ons, Secrets, add GITHUB_TOKEN with a fine-grained PAT that can read this repository, then attach it to this notebook.')
        print('Trying an unauthenticated clone (works only if the repo is public)...')
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', RELEASE_BRANCH,
                       f'https://github.com/{REPO_OWNER}/{REPO_NAME}.git', str(ROOT)],
                      check=True)
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('working directory:', Path.cwd())

## 2. Install and detect hardware

In [ ]:
!pip install -q -e ".[data]"

In [ ]:
import torch
from iridium.runtime.device import detect
from iridium.presets import PRESETS, FREE_TIERS, get_preset, preset_table

info = detect()
DEVICE = info.device
print(info.describe())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

# Kaggle: P100 or 2xT4 depending on what you selected in Settings.
PRESET = 'chat-100m'
preset = get_preset(PRESET)
print()
print(preset_table())

## 3. Dry run -- build, cost, audit, before anything trains

No network, no training: builds the model on the meta device, checks the
instantiated parameter count against `IridiumConfig`'s own formula, and
prints the data-budget audit (`iridium.training.budget.audit`) plus the
free-tier time estimates for every tier in `FREE_TIERS`. Read this before
committing a free session to a long run.

In [ ]:
from iridium.training.run_preset import dry_run

dry_run_result = dry_run(preset)
if not dry_run_result['match']:
    raise RuntimeError(
        f"parameter count mismatch: built {dry_run_result['parameters_built']:,}, "
        f"formula says {dry_run_result['parameters_formula']:,}. Do not train "
        "until this reconciles -- report it rather than proceeding."
    )

## 4. Train in rounds

Checkpoint persistence for this host:

In [ ]:
# /kaggle/working is preserved as this notebook's Output when you
# commit the notebook (Save Version); it is not preserved for an
# interactive-only session that is never committed.
from pathlib import Path
OUT_DIR = Path('/kaggle/working/iridium-runs')
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('checkpoints ->', OUT_DIR, '(persists once you Save Version)')

Training streams real text (and, depending on the preset, chat/tool/media
data) over the network in rounds of fresh data -- see the module docstring
in `iridium/training/run_preset.py` for why rounds exist. Each round saves
a checkpoint; the loop below trains the whole preset unless you lower
`STEPS`/`ROUNDS` for a shorter first try.

In [ ]:
from pathlib import Path
from iridium.presets import with_tokens
from iridium.training.prepare import prepare
from iridium.training.run_preset import train_preset

TOKENS = None         # e.g. 1e9 to override the preset's token budget
RESUME_FROM = 'auto'  # picks up the newest round in OUT_DIR/<preset>; '' forces a fresh run
DATA_DIR = OUT_DIR / 'data'   # tokenized shards; prepare once, reuse across sessions

if TOKENS:
    preset = with_tokens(preset, int(TOKENS))
# Step 1: stream, tokenize and write the language data to disk once.
# Skipped when shards already exist (e.g. prepared on a free CPU session).
if not (DATA_DIR / preset.name / 'prepared.json').exists():
    prepare(preset, DATA_DIR)
# Step 2: train from the memory-mapped shards; RAM use no longer grows
# with the token budget.
checkpoint = train_preset(
    preset, device=DEVICE, out=str(OUT_DIR), data=str(DATA_DIR),
    resume=(RESUME_FROM or None), seed=0,
)
print('final checkpoint:', checkpoint)

## 5. Resuming after a session ends

Every round from section 4 is a checkpoint: `train_preset` saves
`OUT_DIR/<preset>/<preset>-round0.pt`, `-round1.pt`, ... and a final `<preset>-final.pt`. A
free session can end mid-run without warning, and each round's file is
under `/kaggle/working`, so it is there only if you Saved a Version before the session ended.

To continue: set `RESUME_FROM` in the training cell above to the last
`roundN.pt` you have, re-run this notebook from the top (cloning and
installing are idempotent), and re-run the training cell. `train_preset`
restores the optimizer, step count and learning-rate schedule -- the
schedule spans the whole run, so a resumed run is not a fresh one with the
clock reset. It does start a new data shuffle; resuming is not bit-for-bit
replay of the interrupted round.

## 6. Chat with the result

In [ ]:
from iridium.training.trainer import load_checkpoint
from iridium.runtime.chat import ChatSession
from iridium.training.tokenizer_bridge import tokenizer_from_manifest

model, manifest = load_checkpoint(str(checkpoint), device=DEVICE)
chat = ChatSession(model, tokenizer=tokenizer_from_manifest(manifest))
print(chat.send('Hello! What are you, and what can you actually do right now?'))

The same checkpoint works from a terminal, outside this notebook:

```bash
python -m iridium chat --checkpoint $OUT_DIR/<preset>/<preset>-final.pt --device auto
```

(`python -m iridium chat` also defaults to the bundled small demo
checkpoint if you omit `--checkpoint`; that one is a fixed-recipe
fine-tune, not the model this notebook trained.)

## 7. Optional: stage 2, add tool use (`tools-100m`)

Initialises from the chat checkpoint you just trained (`--init`) rather
than from scratch. This stage depends on `iridium.runtime.tools`, which is
being built alongside this notebook; the cell below checks for it and
tells you plainly if it is not there yet rather than failing deep inside a
training run.

In [ ]:
import importlib.util

TOOLS_PRESET = 'tools-100m'

if importlib.util.find_spec('iridium.runtime.tools') is None:
    print(
        'iridium.runtime.tools is not importable in this checkout yet -- '
        'tool use is still landing. Re-clone once it has merged and rerun '
        'this cell; skipping the tools stage for now.'
    )
    tools_checkpoint = None
else:
    tools_preset = get_preset(TOOLS_PRESET)
    tools_checkpoint = train_preset(
        tools_preset, device=DEVICE, out=str(OUT_DIR), init=str(checkpoint), seed=0,
    )
    print('tools checkpoint:', tools_checkpoint)